In [1]:
import csv
import importlib
import os
import random
import sys
import torch
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from time import sleep
from collections import deque, defaultdict
from itertools import count
from typing import Any, Dict, Counter, List

sources_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if sources_path not in sys.path:
    sys.path.append(sources_path)

from importnb import Notebook
with Notebook():
    from Labs.LatencyModel import LatencyModel, MultiDULatencyModel
    from Labs.Policy import DrlPolicy
    from Labs.CacheEngine import CacheEngineEnv
    from Labs.UserRequest import UserRequestEvents
    from Labs.EnvWrapper import EnvWrapper

from RL.Networks import QNetwork, MultiHeadQNetwork
from RL.Buffers import ReplayBuffer, NStepReplayBuffer
from RL.Adapters import FeatureAdapter, NetworkAdapter
from RL.FocusWorkers import BaseWorker, EnhWorker, FocusWorker
from RL.A2CWorker import A2CWorker

import Common.config as config
import Common.datatypes as datatypes
import Common.debugger as debugger
import Common.utils as utils
import Core.builders as builders


importlib.reload(builders)
importlib.reload(config)
importlib.reload(datatypes)
importlib.reload(debugger)
importlib.reload(utils)

<module 'Common.utils' from 'c:\\Users\\es25591\\Workspace\\CacheVideoPredict360\\Sources\\Common\\utils.py'>

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

UserTransition = datatypes.UserTransition
CachePolicy = datatypes.CachePolicy
CacheKey = datatypes.CacheKey

cfg = config.Config()
cfg.filename = \
    f"focus_eps{cfg.epsilon_start}_" \
    f"lrdecay{cfg.learning_rate_decay}_" \
    f"gamma{cfg.gamma}.csv"
cfg.state_dim_base_focus = cfg.cache_size * 10 + 2
cfg.state_dim_enh_focus = cfg.cache_size * 10 + 2
cfg.action_dim_base_focus = cfg.cache_size * 5 + 1

debugger = debugger.debug

In [3]:
class NetworkAdapter:
    def __init__(self, cfg: Any, env: Any, feature_adapter: Any):
        self.env = env
        self.cfg = cfg
        self.features = feature_adapter

        self.C = self.cfg.cache_size  # paper's cache capacity (videos)
        self.k = self.cfg.viewport    # paper's tiles per video (enhancement)

    def build_observation(self, req) -> np.ndarray:
        
        if req is None:
            return (
                np.zeros(self.cfg.state_dim_base_focus, dtype=np.float32), 
                np.zeros((self.k, self.cfg.state_dim_enh_focus), dtype=np.float32)
            )

        video = req["video"]
        viewport = req["viewport"]
        
        cache = self.env.mec_cache.policy.cache

        x_s = np.zeros(len(cache), dtype=np.float32)
        x_l = np.zeros(len(cache), dtype=np.float32)

            
        for idx, (v, t) in enumerate(cache):
            if v == -1:
                continue

            if t == -1:
                x_s[idx] = self.features.video_freq_short.get(v, 0) / self.features.video_hist_short.maxlen
                x_l[idx] = self.features.video_freq_long.get(v, 0) / self.features.video_hist_long.maxlen
            else:
                x_s[idx] = self.features.tile_freq_short.get((v, t), 0) / self.features.tile_hist_short.maxlen
                x_l[idx] = self.features.tile_freq_long.get((v, t), 0) / self.features.tile_hist_long.maxlen

        y_s = np.array(
            [self.features.video_freq_short.get(video, 0) / self.features.video_hist_short.maxlen], dtype=np.float32
        )
        y_l = np.array(
            [self.features.video_freq_long.get(video, 0) / self.features.video_hist_long.maxlen], dtype=np.float32
        )

        state_video = np.concatenate([x_s, x_l, y_s, y_l], axis=0) 
        
        state_vp = []
        for i, tile in enumerate(viewport):
            z_s = np.array(
                [self.features.tile_freq_short.get((video, tile), 0) / self.features.tile_hist_short.maxlen], dtype=np.float32
            )
            z_l = np.array(
                [self.features.tile_freq_long.get((video, tile), 0) / self.features.tile_hist_long.maxlen], dtype=np.float32
            )

            state_vp.append(np.concatenate([x_s, x_l, z_s, z_l], axis=0))
            
        return state_video, state_vp

    def reset(self):
        
        obs, info = self.env.reset()
        self.features.reset_history()

        return obs, info
    
    def env_is_done(self) -> bool:
        return self.env.users_env.all_users_done()

In [4]:
def save_training_results(
    path_,
    filename,
    ep, 
    total_reward, 
    cache_hits, 
    cache_misses, 
    agent
):
    with open(os.path.join(path_, filename), 'a', newline='') as f:
        fieldnames = [
            'episode', 
            'total_reward', 
            'cache_hits', 
            'cache_misses', 
            'epsilon',
            'lr'
        ]
        writer_results = csv.DictWriter(f, fieldnames=fieldnames)

        if ep == 0:
            writer_results.writeheader()
        
        writer_results.writerow({
            'episode': ep,
            'total_reward': round(float(total_reward), 2),
            'cache_hits': cache_hits,
            'cache_misses': cache_misses,
            'lr': f"{agent.scheduler.get_last_lr()[0]:.10f}" if agent else None,
            'epsilon': round(float(agent.epsilon), 4) if agent else None
        })

def update_metrics(info: dict, reward: float) -> tuple[float, int, int, int, int]:
    enh_hits = info.get("enh_layer_hits", 0)
    base_hits = info.get("base_layer_hits", 0)
    enh_misses = info.get("enh_layer_misses", 0)
    base_misses = info.get("base_layer_misses", 0)

    return reward, base_hits, base_misses, enh_hits, enh_misses

In [ ]:
def select_action(state, base_agent, enh_agent, req_state, env):

    if req_state is None:
        return np.zeros(5, dtype=np.int32), np.zeros(5, dtype=np.int32), None

    state_base, state_enh = state

    req = req_state
    missing = env._missing_items(req)

    if missing[0] == 1:
        action_base, value_base, probs_base = base_agent.select_action(state_base)
    else:
        action_base, value_base, probs_base = 0, 0, 0

    transition = [
        {
            'state': state_base,
            'action': action_base,
            'value': value_base,
            'probs': probs_base
        }
    ]

    enh_actions = []

    for idx, missing_item in enumerate(missing[1:]):
        if missing_item == 1:
            action_enh, value_enh, probs_enh = enh_agent.select_action(state_enh[idx])
            enh_actions.append(action_enh)
        else:
            enh_actions.append(0)

        transition.append({
            'state': state_enh[idx],
            'action': action_enh if missing_item == 1 else 0,
            'value': value_enh if missing_item == 1 else None,
            'probs': probs_enh if missing_item == 1 else None
        })

    return np.concatenate([[action_base], enh_actions], axis=0), missing, transition

def run_episode(episode, env, agent, net_adapter, cfg):
    """Run one full training episode."""
    _, info = net_adapter.reset()

    total_reward = 0.0
    cache_hits = cache_misses = 0
    base_hits = base_misses = 0
    enh_hits = enh_misses = 0
    
    videos_req = []

    # if cfg.has_warmup:
    #     env.warmup_phase(net_adapter, 1000)

    for step in range(cfg.max_steps):

        # --- Build State ---
        req_state = info.get("user_request", None)
        state = net_adapter.build_observation(req_state)

        # --- Action Selection --- 
        action, missing, transition = select_action(state, agent, agent, req_state, env)

        # --- Environment Step ---
        _, reward, done, info = env.step(action, req_state, net_adapter)

        # --- Store Transition & Train ---
        nxt_req = info["user_request"]

        reward_0 = info["reward_layer_0"]
        reward_1 = info["reward_layer_1"]
        reward = reward_0 + reward_1

        prefetch_base = info["prefetch_base"]
        prefetch_enh = info["prefetch_enh"]

        state_base, state_enh = state
        next_state_base, next_state_enh = net_adapter.build_observation(nxt_req)

        queued_update = False

        if missing[0] == 1 and prefetch_base:
            agent.remember(
                transition[0]['probs'], 
                transition[0]['value'], 
                reward_0 + reward_1, 
                next_state_base, 
                done
            )
            queued_update = True

        for i in range(len(state_enh)):
            if missing[i + 1] == 1 and prefetch_enh:
                agent.remember(
                    transition[i + 1]['probs'], 
                    transition[i + 1]['value'], 
                    reward_0 + reward_1, 
                    next_state_enh[i], 
                    done
                )
                queued_update = True

        if queued_update:
            agent.train_step()

        delta_r, bs_hits, bs_miss, e_hits, e_miss = update_metrics(info, reward)
        total_reward += delta_r
        cache_hits += bs_hits + e_hits
        cache_misses += bs_miss + e_miss
        base_hits += bs_hits
        base_misses += bs_miss
        enh_hits += e_hits
        enh_misses += e_miss

        if done:
            break

        debugger.log('cache_hits', bs_hits + e_hits)
        debugger.log('cache_misses', bs_miss + e_miss)

        nxt_req_state = info["user_request"]

        # if nxt_req_state['video'] is not None:
        #     videos_req.append(nxt_req_state['video'])

        # if step % 50 == 0 and (episode + 1) % 10 == 0:

        #     counter_videos = Counter(videos_req)
        #     most_common_videos = counter_videos.most_common(5)
            
        #     print(f"Cache postions: {env.mec_cache.policy.cache}")
        #     print(f"Counter videos: {counter_videos}")
        #     print(f"\nMost requested videos so far: {most_common_videos}")
        #     print(f"Total videos requested so far: {len(videos_req)}")
        #     print(f"Total unique videos requested: {len(counter_videos)}")
        #     print(f"videos_req: {videos_req[-20:]}")
        #     print(f"base_action: {debugger.data.get('base_action', 'N/A')[-20:]}")
        #     print(f"base_layer_miss: {debugger.data['base_layer_miss'][-20:]}")
        #     print(f"\nEpisode {episode} | Step {step} | Reward: {delta_r:.2f} | Total Reward: {total_reward:.2f}" )
        #     print(debugger)

        #     # input("Press Enter to continue to the next step...")
        #     print("-" * 5)

    return total_reward, cache_hits, cache_misses, base_hits, base_misses, enh_hits, enh_misses

def train(cfg):
    env = builders.build_environment(cfg)

    # agent = FocusWorker(cfg, debugger=debugger)
    
    agent = A2CWorker(cfg, debugger=debugger)

    feature_adapter = FeatureAdapter(cfg, env)
    net_adapter = NetworkAdapter(cfg, env, feature_adapter)

    date_dir = pd.Timestamp.now().strftime("%Y-%m-%d_%H-%M")
    debug_path = os.path.join(cfg.path_results, date_dir)
    os.makedirs(debug_path, exist_ok=True)

    print(f"Starting training for {cfg.n_episodes} episodes... {date_dir}")
    print(f"Warmup Phase: {'Enabled' if cfg.has_warmup else 'Disabled'}")
    print(f"Users Session Length: {cfg.user_session_length}")
    print(
        f"State Dim Base: {agent.state_dim}, Action Dim Base: {agent.action_dim}, Hidden Dim Base: {cfg.hidden_dim_base_focus}"
    )

    for episode in range(cfg.n_episodes):

        total_reward, hits, misses, bs_hits, bs_miss, enh_hits, enh_miss = run_episode(
            episode, env, agent, net_adapter, cfg
        )

        agent.update_epsilon()

        save_training_results(
            path_=cfg.path_results + "/" + date_dir,
            filename=cfg.filename,
            ep=episode,
            total_reward=total_reward,
            cache_hits=hits,
            cache_misses=misses,
            agent=agent
        )

        debugger.log('lr', agent.scheduler.get_last_lr()[0])
        debugger.log('epsilon', agent.epsilon)

        print(
            f"--- Episode {episode} | R: {total_reward:.2f} | "
            f"HR: {hits / (hits + misses + 1e-9):.2f} | "
            f"BHR: {bs_hits / (bs_hits + bs_miss + 1e-9):.2f} | "
            f"EHR: {enh_hits / (enh_hits + enh_miss + 1e-9):.2f} ---"
        )

        debugger.save_results(filepath=f"{debug_path}/debug_ep{episode}")
        debugger.clear()

        print("-" * 50)

if __name__ == "__main__":
    train(cfg)

Starting training for 400 episodes... 2026-03-20_13-29
Warmup Phase: Enabled
Users Session Length: 60
State Dim Base: 502, Action Dim Base: 251, Hidden Dim Base: 256
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0


c:\Users\es25591\Workspace\CacheVideoPredict360\Sources\RL\A2CWorker.py:88: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  next_state = torch.FloatTensor(next_state).to(self.device)


0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0


KeyboardInterrupt: 